# A stellarator run through `functional_process`, on the current API

`functional_process` ports PROCESS's stellarator models into cottax (`~/jaxgraph`): a
**declared graph** of nodes with typed ports, decomposed into blocks, each block driven
by an explicit, autodiff-visible algorithm — instead of PROCESS's own architecture, which
treats the whole pipeline as one opaque function and differentiates it by finite
differences.

This is a short walkthrough, not the full tour (see git history for a longer earlier
version). It assembles the graph for one machine — the Helias stellarator of
`tests/regression/input_files/stellarator_helias.IN.DAT` — solves it, and then spends
most of its time on one fact: **the graph is now a value**. A freshly re-assembled graph
is a new object, but it compares *equal* to the first one, and JAX's own cache is keyed
on equality, not identity. Re-assembling from scratch and solving again should therefore
compile nothing.

Everything below actually executes; runs top to bottom in a fresh kernel.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)

import os
import time
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "functional_process").is_dir())
os.chdir(REPO)

import inspect
import subprocess
import sys
from collections import Counter

import jax.numpy as jnp
import numpy as np
from cottax.blocking import Blocking

from functional_process import boundary, mda, session
from functional_process.indat import REFERENCE_INPUT_FILE, graph_for, machine_from_indat

SCRATCH = Path("/tmp/claude-1000/-home-tbogaarts-PROCESS/031c3ec8-ef1e-47a6-841c-41a7b5e49c67/scratchpad/nbwork")
SCRATCH.mkdir(parents=True, exist_ok=True)
PY = sys.executable

print(REPO, "|", jnp.zeros(1).dtype, "|", REFERENCE_INPUT_FILE, "|", PY)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


/home/tbogaarts/PROCESS/.claude/worktrees/agent-ae8399e0bd9e69d82 | float64 | tests/regression/input_files/stellarator_helias.IN.DAT | /home/tbogaarts/miniconda/envs/process_port/bin/python


## 1. Assemble the graph, and look at it

`machine_from_indat` reads only the input file's integer switches and builds a tree of
model instances; `graph_for` walks that tree into a cottax `Graph` -- a binding of node
*places* to *definitions*. Nothing runs yet.

In [2]:
machine = machine_from_indat(REFERENCE_INPUT_FILE)
graph = graph_for(machine)
print(type(machine).__name__, "->", len(graph.nodes), "nodes")
print(Counter(type(d).__name__ for d in graph.definitions.values()))

StellaratorProcess -> 154 nodes
Counter({'ImplementedFunction': 150, 'RootFind': 2, 'FixedPoint': 2})


`ImplementedFunction` is the ordinary node -- ports plus a body (`CallableNode` in the old
API). `RootFind`/`FixedPoint` are the self-loops PROCESS solves by re-running a model
until it stops moving, declared as problems rather than bodies.

One node, opened up: `In`/`Out` ports are `VarPath`s into PROCESS's own namespace, and the
function that computes them.

In [3]:
place = next(n for n in graph.nodes if n.path_str() == ".stellarator.sudo_density_limit")
node = graph.definitions[place]
print(type(node).__name__, "at", place.path_str())
print("  reads:", [v.path_str() for v in node.reads])
print("  owns :", [v.path_str() for v in node.owns])

ImplementedFunction at .stellarator.sudo_density_limit
  reads: ['.physics.b_plasma_toroidal_on_axis', '.physics.p_plasma_loss_mw', '.physics.rmajor', '.physics.rminor', '.physics.nd_plasma_electrons_vol_avg', '.physics.nd_plasma_electron_line']
  owns : ['.physics.nd_plasma_electrons_max']


### Decomposed into blocks, each driven by an algorithm chosen for it

`mda.driven_graph` cuts the raw cross-node cycles into declared problems and attaches a
driver to each; `Blocking.scc` condenses the result into strongly connected components.
Most blocks are a single node, run once, in a derived order -- only the genuinely coupled
blocks are driven by an iterative algorithm.

In [4]:
driven = mda.driven_graph(graph)
blocking = Blocking.scc(driven)
n_driven = sum(1 for t in blocking.problem_types if t is not None)
print(f"{len(blocking.blocks)} blocks, {n_driven} of them driven")
print("block sizes:", sorted(Counter(len(b) for b in blocking.blocks).items()))

144 blocks, 6 of them driven
block sizes: [(1, 138), (2, 4), (3, 1), (7, 1)]


And the boundary -- every read that is not produced by another node in the graph.
`arrays are refused in the graph` (`CLAUDE.md`): what used to be `carried` fields is now
a `stated` port, an output of a source node living in the env rather than baked into a
declaration -- a third boundary category alongside `input` (a genuine external read) and
`guess` (a `Start` port minted for a driven unknown).

In [5]:
rows = boundary.boundary(driven)
print(Counter(kind for kind, _ in rows))

Counter({'input': 289, 'stated': 16, 'guess': 6})


## 2. Solve it, against PROCESS

`session.open_session(..., mode="provider")` runs PROCESS's own `SingleRun` once (cached
to disk after the first time) to get both a cold starting `DataStructure` and PROCESS's
own converged answer to compare against. `Session.mdf` assembles the MDF arm on first
call and solves it -- PROCESS's own architecture (optimise the design, converge the
coupled models inside every evaluation), same SQP (`pyvmcon`), same convergence test,
gradients from one `jax.jacfwd` instead of a finite-difference pipeline sweep per
iteration variable.

In [6]:
import contextlib
import io

from process.core.solver.objectives import objective_function

with contextlib.redirect_stdout(io.StringIO()):   # PROCESS's own console noise
    live = session.open_session(REFERENCE_INPUT_FILE, mode="provider")
reference = live.reference
print(f"PROCESS: {reference.solver_iterations} VMCON iterations in "
      f"{reference.solve_seconds:.1f} s")

began = time.perf_counter()
answer = live.mdf()
first_solve_seconds = time.perf_counter() - began
print(f"port MDF: {answer['iterations']} SQP iterations in {first_solve_seconds:.1f} s")

process_objf = objective_function(reference.i_figure_merit, reference.data)
port_objf = answer["objf"]
print(f"\nobjf   port {port_objf:.9f}   PROCESS {process_objf:.9f}   "
      f"rel {abs(port_objf - process_objf) / abs(process_objf):.2e}")

worst = max(
    abs(x - reference.converged[i]) / abs(reference.converged[i])
    for i, x in zip(reference.ixc, answer["_x"])
)
print(f"worst relative deviation on any of the 8 design variables: {worst:.2e}")

PROCESS: 46 VMCON iterations in 98.4 s
port MDF: 41 SQP iterations in 19.0 s

objf   port 1.218482841   PROCESS 1.214916785   rel 2.94e-03
worst relative deviation on any of the 8 design variables: 1.09e-01


The objective agrees closely; the design vector mostly does too. (One variable sits on a
kink in the model -- a clamped square root where PROCESS's finite difference sees a wide
chord and autodiff sees the exact one-sided slope -- already diagnosed, not a port
defect. Not chased further here.)

## 3. The centrepiece: re-assemble from scratch, and watch it compile nothing

`graph_for(machine_from_indat(...))` run twice gives two different Python objects. Are
they the same graph? Under the old API this question needed an explicit `==`; under the
new one it also decides whether JAX's own executable cache -- keyed by equality, not
identity -- treats the second assembly as new work at all.

In [7]:
machine_a = machine_from_indat(REFERENCE_INPUT_FILE)
machine_b = machine_from_indat(REFERENCE_INPUT_FILE)
graph_a, graph_b = graph_for(machine_a), graph_for(machine_b)
print("equal:", graph_a == graph_b, "  same object:", graph_a is graph_b)

equal: True   same object: False


Measured in a **fresh subprocess** (so "first assembly" is genuinely cold, not warmed by
section 2's own solve above): assemble MDF, seed, prime and solve; then do the whole
thing again **from scratch**, twice -- a second and third
`graph_for(machine_from_indat(...))` and `build_mdf`, on the same graph *content* but not
the same objects.

In [8]:
gate_script = SCRATCH / "reassembly_gate.py"
gate_script.write_text(r"""
import time
import traceback

import jax

jax.config.update("jax_enable_x64", True)
from jax._src import compiler

from functional_process import session
from functional_process.run_cold_matrix import build_mdf, solve_mdf
from functional_process.indat import REFERENCE_INPUT_FILE, graph_for, machine_from_indat

# `session._Compiles` counts XLA compilations by wrapping this same entry point; this
# also records *what* was compiled and what it cost, off the stack under it.
RECORD = []
_compile = compiler.backend_compile_and_load

def counted(*args, **kwargs):
    frames = [f"{f.filename.rsplit('/', 1)[-1]}:{f.name}" for f in traceback.extract_stack()]
    # `apply_primitive` first, and the order matters: a one-primitive eager dispatch
    # that happens *inside* a `host_cache` call has both markers on its stack, and it
    # is the eager dispatch that is being compiled.
    kind = ("eager one-op" if any("apply_primitive" in f for f in frames)
            else "host_cache" if any("host_cache" in f for f in frames)
            else "whole-schedule jit" if any("run_schedule" in f for f in frames)
            else "other")
    began = time.perf_counter()
    try:
        return _compile(*args, **kwargs)
    finally:
        RECORD.append((kind, time.perf_counter() - began))

compiler.backend_compile_and_load = counted

live = session.open_session(REFERENCE_INPUT_FILE)
reference, cold = live.reference, live.cold

def assemble_and_solve():
    machine_graph = graph_for(machine_from_indat(REFERENCE_INPUT_FILE))
    build = build_mdf(reference, machine_graph, None, root_find=False)
    return solve_mdf(build, reference, cold)

xs = []
for label in ("first assembly", "re-assembled from scratch", "re-assembled again"):
    RECORD.clear()
    began = time.perf_counter()
    result = assemble_and_solve()
    seconds = time.perf_counter() - began
    xs.append(result["_x"])
    print(f"ROW|{label}|{seconds}|{len(RECORD)}")
    for kind in ("host_cache", "whole-schedule jit", "eager one-op", "other"):
        paid = [dt for k, dt in RECORD if k == kind]
        if paid:
            print(f"WHAT|{label}|{kind}|{len(paid)}|{sum(paid)}")
print(f"SAME|{all(x == xs[0] for x in xs)}")
""")

# `cwd=REPO` is not enough. A subprocess puts the *script's* directory on `sys.path[0]`,
# not its working directory, so `functional_process` would be resolved through the
# editable install -- which points at the main checkout and not necessarily at the tree
# this notebook is running out of. Inside a git worktree those are different files.
env = {**os.environ, "PYTHONPATH": str(REPO)}
out = subprocess.run([PY, str(gate_script)], cwd=REPO, env=env,
                     capture_output=True, text=True, check=True)
lines = out.stdout.splitlines()

for line in lines:
    if line.startswith("ROW|"):
        _, label, seconds, n_compiles = line.split("|")
        print(f"{label:<28s} {float(seconds):6.2f} s   {n_compiles} compiles")

print("\nwhat the first assembly's compiles were:")
for line in lines:
    if line.startswith("WHAT|first assembly|"):
        _, _, kind, n, seconds = line.split("|")
        print(f"  {kind:<20s} {n:>3s} programs   {float(seconds):5.2f} s")

same = next(l for l in lines if l.startswith("SAME|")).split("|")[1]
print("\nsame answer every time:", same)

first assembly                19.25 s   29 compiles
re-assembled from scratch      1.16 s   0 compiles
re-assembled again             0.92 s   0 compiles

what the first assembly's compiles were:
  host_cache             2 programs    6.99 s
  whole-schedule jit     2 programs    4.39 s
  eager one-op          25 programs    0.18 s

same answer every time: True


**Measured on this tree: 29 compiles / 19.3 s -> 0 compiles / 1.2 s, and 0 again on a
third assembly.** Structural equality of the re-assembled graph is enough: JAX's own
cache hits every one of the 29 programs, and a re-assembly pays only trace, dispatch and
arithmetic. That is the claim this section exists to check, and it holds.

The 29 are four programs and a long tail. The two `host_cache` ones are what the SQP
calls at every iteration -- `values` and `jacobian`; the fused third is built but never
traced, because `VmconDriver.fused` is off -- and the two whole-schedule ones are the
single `equinox.filter_jit` over the MDA schedule, traced once for `mdf.prime`'s env and
once for `mdf.solve`'s tail. Between them they are 11.4 s of the 19.3. The other 25 are
one-primitive *eager* dispatches (`jnp.asarray` on a Python scalar, a `ravel_pytree`
reshape) that XLA compiles as their own tiny modules: 0.18 s for all of them together,
and cached process-globally rather than per graph.

**It read `29 -> 2` the first time this notebook ran, and finding the two was worth the
detour.** They were the whole-schedule jit. `sand_harness._SCHEDULE_WHOLE` memoises that
jit on the `Schedule`, so a re-assembled schedule has to compare *equal* to reuse the
compiled program -- and it did not, on exactly one leaf of the whole graph:
`SeededNewtonDriver.seed`, produced by a factory (`mda._root_find_seed(problem)`) whose
`problem` argument its body never read. A fresh closure every assembly, and a frozen
`equinox.Module` compares a function field by identity. Hoisting that body to module
level is the entire fix. It is `_audit/optimise_design.md` §37's own lesson -- *a function
built inside a call is a new cache key every call* -- surviving in the one place §37 did
not look.

Which is also why §37.4's table reads **3 compiles / 19.29 s -> 0 / 0.24 s** rather than
`29 -> 0`: it measures a deliberately narrower repro -- assemble, seed, `bind`, and call
the three `host_cache` programs -- with no `prime`, no SQP solve, and therefore no
`run_schedule` anywhere in it. Its `0` was true of what it measured; this notebook's
whole-solve repro simply reaches further, and had to be made true of that too.

## 4. The three levels of "warm"

Same solve (`stellarator_helias`, MDF), three regimes:

- **cold** -- fresh process, no persistent cache: JAX traces every jitted block, lowers
  it to HLO, compiles it to native code, and loads the executable. All four steps paid.
- **persistent disk cache** (`--cache`, `run_cold_matrix._enable_compilation_cache`) --
  a fresh process again, so trace and lower still happen, but the *compile* step is a
  cache hit against `jax_compilation_cache_dir` and is skipped.
- **same process, objects retained** -- no fresh process at all. The Python-level jit
  cache already holds the compiled executable keyed on this exact (structurally equal)
  graph, so nothing above the arithmetic runs again.

Each regime is a full solve of the same problem via `session.open_session(...).mdf()` --
subprocesses for the first two, so "fresh process" is real and not simulated.

In [9]:
driver_script = SCRATCH / "warm_level.py"
driver_script.write_text(r'''
import argparse, time
parser = argparse.ArgumentParser()
parser.add_argument("--cache", default=None)
args = parser.parse_args()
import jax
jax.config.update("jax_enable_x64", True)
if args.cache:
    from functional_process.run_cold_matrix import _enable_compilation_cache
    _enable_compilation_cache(args.cache)
from functional_process import session
from functional_process.indat import REFERENCE_INPUT_FILE
t0 = time.perf_counter()
live = session.open_session(REFERENCE_INPUT_FILE)
answer = live.mdf()
t1 = time.perf_counter()
print(f"SECONDS {t1 - t0:.3f}")
''')

def run_subprocess(cache_dir=None):
    args = [PY, str(driver_script)]
    if cache_dir is not None:
        args += ["--cache", str(cache_dir)]
    # `PYTHONPATH` for the reason section 3's subprocess needed it: a subprocess
    # resolves imports off the script's directory, not off `cwd`.
    out = subprocess.run(args, cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO)},
                         capture_output=True, text=True, check=True)
    line = next(l for l in out.stdout.splitlines() if l.startswith("SECONDS"))
    return float(line.split()[1])

cache_dir = SCRATCH / "jax_persistent_cache"
import shutil
shutil.rmtree(cache_dir, ignore_errors=True)
cache_dir.mkdir(parents=True)

cold_seconds = run_subprocess(cache_dir=None)
_populate_seconds = run_subprocess(cache_dir=cache_dir)   # fills the cache, not reported
warm_disk_seconds = run_subprocess(cache_dir=cache_dir)

live2 = session.open_session(REFERENCE_INPUT_FILE)
_ = live2.mdf()                      # pays the compile, in this process
began = time.perf_counter()
_ = live2.mdf()                      # objects retained -- everything above is cached
warm_process_seconds = time.perf_counter() - began

print(f"{'cold':<24s} {cold_seconds:7.2f} s")
print(f"{'persistent disk cache':<24s} {warm_disk_seconds:7.2f} s")
print(f"{'same process, retained':<24s} {warm_process_seconds:7.2f} s")

cold                       19.38 s
persistent disk cache       9.24 s
same process, retained      0.76 s


**cold** pays all four steps: trace the Python into a jaxpr, lower that to HLO, hand HLO
to XLA's backend compiler, load the resulting executable. **Persistent disk cache** skips
only the compile step -- tracing and lowering still happen in the fresh process, XLA just
finds a matching compiled artifact on disk instead of building one -- which is why it is
roughly half of cold here rather than near-zero. **Same process, retained** skips all four:
the Python-level jit cache already holds the compiled executable for this exact
(structurally-equal) graph, so calling `.mdf()` again is dispatch and arithmetic, nothing
JAX-internal at all -- consistent with section 3's cache hit and with `session.py`'s own
reasoning for why re-assembly, not caching, is the trap (section 5).

## 5. The trap, and how much of it is left

Re-assembling *between* solves throws away level 3, every time. That used to be the
difference between a second and a minute: `_audit/optimise_design.md` §32.2 measured a
re-assembling loop at **30-75x** the cost of one that keeps its objects, because every
re-assembly recompiled the whole graph. Since the graph became a value that cost is gone
-- section 3's "re-assembled from scratch" row is 1.16 s against level 3's 0.76 s, and
the ~0.4 s between them is assembling, seeding and priming, not compilation.

So the advice survives with a much smaller number attached to it.
`run_cold_matrix.run_one` re-assembles per call and is still the wrong loop to call
twice; the fast path is `functional_process/session.py` -- assemble a `Session` once,
then `.mdf(cold=...)`/`.sand(cold=...)` per point, which only re-seeds and re-primes
(10-40 ms, zero compiles, per that module's own measurements). What changed is that
re-assembly is now a ~1.5x tax on a solve rather than a trap that cannot be afforded.